# INCEpTION gold corpus — reproducibility (D)

This notebook asks whether `data/gcn_gold_corpus/` is a deterministic function of the
INCEpTION export at `data/inception/project-2026-08-08-072803/`: it regenerates both the
flatten and normalise stages from scratch, into a throwaway directory, and compares every
byte against what is already on disk. Notebooks A, B and C each treat one side of the
pipeline as ground truth and never open the export directly; this notebook is different
because its job is to test whether that ground truth reproduces at all, which requires
reading the export as an independent third source rather than trusting either derived table.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 250)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/interim/gcn_gold_corpus").is_dir())
EXPORT_ROOT = ROOT / "data/inception/project-2026-08-08-072803"
INTERIM_DIR = ROOT / "data/interim/gcn_gold_corpus"
CORPUS_DIR = ROOT / "data/gcn_gold_corpus"
TABLE_NAMES = ["documents", "annotators", "evidence_spans", "photometry_spans", "event_summaries"]


def sha256_of(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


rows = []
for stage, directory in [("interim", INTERIM_DIR), ("corpus", CORPUS_DIR)]:
    for name in TABLE_NAMES:
        path = directory / f"{name}.parquet"
        df = pd.read_parquet(path)
        rows.append({"file": f"{stage}/{name}.parquet", "rows": len(df), "columns": df.shape[1],
                     "sha256": sha256_of(path)})

baseline_fingerprint = pd.DataFrame(rows)
print(baseline_fingerprint.to_string(index=False))

                            file  rows  columns                                                           sha256
       interim/documents.parquet    10        5 ccae3bdd48b37cdf70cd9528721f8899aa8861f8728cb669070559d284439e84
      interim/annotators.parquet    28       10 21709368263a4b9619818c33d93913fbef8a3d7c254aec956df05d501bb6d289
  interim/evidence_spans.parquet  6610       12 d26b238e5da09139902ff408da5bc6753fb3a8d201315610832c857ec7c645d8
interim/photometry_spans.parquet  2741       22 ea8595165f8f5ef699777af503716ae663b9afdde46c475e9a9964b99ab1233b
 interim/event_summaries.parquet    28       26 bb159056d91458262e84cac21de072a3333920fc1f1881391032f66acb547436
        corpus/documents.parquet    10        5 ccae3bdd48b37cdf70cd9528721f8899aa8861f8728cb669070559d284439e84
       corpus/annotators.parquet    28       10 21709368263a4b9619818c33d93913fbef8a3d7c254aec956df05d501bb6d289
   corpus/evidence_spans.parquet  6615       20 e0f9ad82ada617ac41cfc082769871c3175d08702cd5ffa5

In [2]:
import shutil
import subprocess
import tempfile

TMP_ROOT = Path(tempfile.mkdtemp(prefix="maforai_gold_repro_", dir="/tmp"))
(TMP_ROOT / "scripts" / "gcn_gold").mkdir(parents=True)
(TMP_ROOT / "data").mkdir()
shutil.copy2(ROOT / "scripts/gcn_gold/01_flatten.py", TMP_ROOT / "scripts/gcn_gold/01_flatten.py")
shutil.copy2(ROOT / "scripts/gcn_gold/02_normalise.py", TMP_ROOT / "scripts/gcn_gold/02_normalise.py")
(TMP_ROOT / "data" / "inception").symlink_to(ROOT / "data/inception", target_is_directory=True)

PYTHON = str(ROOT / ".venv" / "bin" / "python")
for script in ["01_flatten.py", "02_normalise.py"]:
    result = subprocess.run([PYTHON, str(TMP_ROOT / "scripts/gcn_gold" / script)],
                            capture_output=True, text=True, cwd=TMP_ROOT)
    print(f"{script}: return code {result.returncode}")
    assert result.returncode == 0, f"{script} failed:\n{result.stderr}"

regen_rows = []
for stage, directory in [("interim", TMP_ROOT / "data/interim/gcn_gold_corpus"),
                         ("corpus", TMP_ROOT / "data/gcn_gold_corpus")]:
    for name in TABLE_NAMES:
        regen_path = directory / f"{name}.parquet"
        on_disk_hash = baseline_fingerprint.loc[
            baseline_fingerprint["file"] == f"{stage}/{name}.parquet", "sha256"].iloc[0]
        regen_hash = sha256_of(regen_path)
        regen_rows.append({"table": name, "stage": stage, "sha256_on_disk": on_disk_hash,
                           "sha256_regenerated": regen_hash, "match": on_disk_hash == regen_hash})

regeneration_check = pd.DataFrame(regen_rows)
print(regeneration_check.to_string(index=False))

01_flatten.py: return code 0


02_normalise.py: return code 0
           table   stage                                                   sha256_on_disk                                               sha256_regenerated  match
       documents interim ccae3bdd48b37cdf70cd9528721f8899aa8861f8728cb669070559d284439e84 ccae3bdd48b37cdf70cd9528721f8899aa8861f8728cb669070559d284439e84   True
      annotators interim 21709368263a4b9619818c33d93913fbef8a3d7c254aec956df05d501bb6d289 21709368263a4b9619818c33d93913fbef8a3d7c254aec956df05d501bb6d289   True
  evidence_spans interim d26b238e5da09139902ff408da5bc6753fb3a8d201315610832c857ec7c645d8 d26b238e5da09139902ff408da5bc6753fb3a8d201315610832c857ec7c645d8   True
photometry_spans interim ea8595165f8f5ef699777af503716ae663b9afdde46c475e9a9964b99ab1233b ea8595165f8f5ef699777af503716ae663b9afdde46c475e9a9964b99ab1233b   True
 event_summaries interim bb159056d91458262e84cac21de072a3333920fc1f1881391032f66acb547436 bb159056d91458262e84cac21de072a3333920fc1f1881391032f66acb547436   Tr

In [3]:
diff_rows = []
failed = regeneration_check[~regeneration_check["match"]]
if failed.empty:
    print("No hash differences found: every regenerated table is byte-identical to what is on disk.")
else:
    for _, f in failed.iterrows():
        stage, name = f["stage"], f["table"]
        on_disk_dir = INTERIM_DIR if stage == "interim" else CORPUS_DIR
        regen_dir = TMP_ROOT / ("data/interim/gcn_gold_corpus" if stage == "interim" else "data/gcn_gold_corpus")
        a = pd.read_parquet(on_disk_dir / f"{name}.parquet")
        b = pd.read_parquet(regen_dir / f"{name}.parquet")
        print(f"{stage}/{name}: on-disk shape {a.shape}, regenerated shape {b.shape}")
        col_diff = sorted(set(a.columns) ^ set(b.columns))
        if col_diff:
            diff_rows.append({"stage": stage, "table": name, "column": str(col_diff), "row_key": "N/A",
                              "on_disk_value": "column set differs", "regenerated_value": "column set differs"})
        shared = [c for c in a.columns if c in b.columns]
        for i in range(min(len(a), len(b))):
            if len(diff_rows) >= 20:
                break
            for c in shared:
                va, vb = a.iloc[i][c], b.iloc[i][c]
                if not (va == vb or (pd.isna(va) and pd.isna(vb))):
                    diff_rows.append({"stage": stage, "table": name, "column": c, "row_key": i,
                                      "on_disk_value": va, "regenerated_value": vb})
                    if len(diff_rows) >= 20:
                        break

difference_localisation = pd.DataFrame(diff_rows, columns=["stage", "table", "column", "row_key",
                                                            "on_disk_value", "regenerated_value"])
if len(difference_localisation):
    print(difference_localisation.to_string(index=False))
else:
    print(f"\n(no differences to localise; empty frame with columns {list(difference_localisation.columns)})")
    print(difference_localisation)

No hash differences found: every regenerated table is byte-identical to what is on disk.

(no differences to localise; empty frame with columns ['stage', 'table', 'column', 'row_key', 'on_disk_value', 'regenerated_value'])
Empty DataFrame
Columns: [stage, table, column, row_key, on_disk_value, regenerated_value]
Index: []


In [4]:
import zipfile
from xml.etree import ElementTree as ET

XMI_NS = "http://www.omg.org/XMI"
CAS_NS = "http:///uima/cas.ecore"
CUSTOM_NS = "http:///webanno/custom.ecore"


def qtag(uri, local):
    return f"{{{uri}}}{local}"


def read_zip_xmi(document, stem):
    with zipfile.ZipFile(EXPORT_ROOT / "annotation" / document / f"{stem}.zip") as zf:
        raw = zf.read(f"{stem}.xmi")
    return ET.fromstring(raw)


def spans_at(root, layer_local, begin, end):
    return [e for e in root.findall(qtag(CUSTOM_NS, layer_local))
           if e.get("begin") == str(begin) and e.get("end") == str(end)]


def show_xml(label, elems):
    if not elems:
        print(f"  {label}: ABSENT (no element at this offset)")
    for e in elems:
        print(f"  {label}: {ET.tostring(e, encoding='unicode').strip()}")


def show_rows(label, df, document, layer_source, begin, end):
    row = df[(df["document_name"] == document) & (df["layer_source"] == layer_source)
            & (df["begin"] == begin) & (df["end"] == end)]
    print(f"  {label} ({len(row)} row(s)):")
    print(row.to_string(index=False) if len(row) else "    (no row)")


interim_ev = pd.read_parquet(INTERIM_DIR / "evidence_spans.parquet")
interim_ph = pd.read_parquet(INTERIM_DIR / "photometry_spans.parquet")
corpus_ev = pd.read_parquet(CORPUS_DIR / "evidence_spans.parquet")
corpus_ph = pd.read_parquet(CORPUS_DIR / "photometry_spans.parquet")
INTERIM_TABLES = {"evidence": interim_ev, "photometry": interim_ph}
CORPUS_TABLES = {"evidence": corpus_ev, "photometry": corpus_ph}


def trace(document, layer_local, table, points, decisions_note):
    for layer_source, begin, end in points:
        show_xml(f"export {layer_source} [{begin}:{end}]",
                 spans_at(read_zip_xmi(document, layer_source), layer_local, begin, end))
    for layer_source, begin, end in points:
        show_rows(f"interim {layer_source} [{begin}:{end}]", INTERIM_TABLES[table], document, layer_source, begin, end)
    for layer_source, begin, end in points:
        show_rows(f"corpus {layer_source} [{begin}:{end}]", CORPUS_TABLES[table], document, layer_source, begin, end)
    print(f"  decisions: {decisions_note}")


print("--- Record 1: photometry span identical to baseline, xmi_id differs ---")
trace("event_2025aji.xmi", "PHOTOMETRIC_MEASUREMENT", "photometry",
     [("INITIAL_CAS", 5599, 5626), ("Camille", 5599, 5626)],
     "1 (span key ignores xmi_id: baseline 159709 vs Camille 159669), 2 (matched on exact "
     "begin/end/span_index), 3 (match_status=accepted), 4 (no feature differs)")

print("\n--- Record 2: a duplicate offset group, both rows ---")
trace("event_GCN-251222_170549.xmi", "PHOTOMETRIC_MEASUREMENT", "photometry",
     [("INITIAL_CAS", 57301, 57338), ("Priyadarshini", 57301, 57338)],
     "1/7 (span_index 0 and 1 disambiguate two Priyadarshini spans at one offset); span_index=0 "
     "matches the baseline (corrected, shares xmi_id 138023 with it), span_index=1 has no baseline "
     "counterpart (created, its own xmi_id 138502)")

print("\n--- Record 3: a span corrected on eight features ---")
trace("event_GRB241030.xmi", "PHOTOMETRIC_MEASUREMENT", "photometry",
     [("INITIAL_CAS", 16299, 16346), ("Dahlia", 16299, 16346)],
     "2 (matched on exact offsets, xmi_id 164011 shared with baseline), 3 (corrected), 4/5 (8 "
     "features differ: photometric_system, photometric_band, obs_time_raw, obs_time_type, "
     "obs_time_reference, exposure_time_raw, instrument, comment)")

print("\n--- Record 4: a baseline span an annotator did not carry forward ---")
trace("event_2025aji.xmi", "ASTRO_EVIDENCE", "evidence",
     [("INITIAL_CAS", 54079, 54089), ("Camille", 54079, 54089)],
     "2 (Camille's export layer has no element at this offset, confirmed above), 3 (a synthetic "
     "'deleted' row is added: xmi_id null, span_index 0, label null, covered_text copied from the "
     "baseline), 6/10 (has_category is set True vacuously on this row)")

print("\n--- Record 5: an annotator note (created, no category, comment) ---")
trace("event_2026owq.xmi", "ASTRO_EVIDENCE", "evidence",
     [("INITIAL_CAS", 52089, 52096), ("Sarah", 52089, 52096)],
     "2 (no baseline element at this offset, confirmed above), 3 (created), 9 (has_category=False "
     "and comment is non-blank -> is_annotator_note=True), 10 (has_category=False)")

--- Record 1: photometry span identical to baseline, xmi_id differs ---
  export INITIAL_CAS [5599:5626]: <ns0:PHOTOMETRIC_MEASUREMENT xmlns:ns0="http:///webanno/custom.ecore" xmlns:ns1="http://www.omg.org/XMI" ns1:id="159709" sofa="1" begin="5599" end="5626" measurement_type="upper_limit" photometric_system="unknown" target="counterpart" certainty="confirmed" magnitude_or_limit="20.7" magnitude_error="" limit_sigma="" unit="mag" photometric_band="" obs_time_raw="240 sec after trigger" obs_time_type="relative_to_trigger" obs_time_reference="trigger_time_t0" exposure_time_raw="" timezone_raw="" instrument="" comment="Photometric band not found; the annotator must confirm it. Photometric system is unknown; verify AB or Vega." />
  export Camille [5599:5626]: <ns0:PHOTOMETRIC_MEASUREMENT xmlns:ns0="http:///webanno/custom.ecore" xmlns:ns1="http://www.omg.org/XMI" ns1:id="159669" sofa="1" begin="5599" end="5626" measurement_type="upper_limit" photometric_system="unknown" target="counterpart

  corpus INITIAL_CAS [5599:5626] (1 row(s)):


    document_name layer_source  xmi_id  begin  end                covered_text measurement_type photometric_system      target certainty magnitude_or_limit magnitude_error limit_sigma unit photometric_band          obs_time_raw       obs_time_type obs_time_reference exposure_time_raw timezone_raw instrument                                                                                                      comment  span_index match_status changed_fields  is_overlapping  has_category  is_annotator_note     comment_status  magnitude_or_limit_numeric  magnitude_error_numeric  limit_sigma_numeric  exposure_time_raw_numeric  extractor_vocabulary_gap
event_2025aji.xmi  INITIAL_CAS  159709   5599 5626 upper limit up to  20.7 mag      upper_limit            unknown counterpart confirmed               20.7                              mag                  240 sec after trigger relative_to_trigger    trigger_time_t0                                           Photometric band not found; the annota

  export INITIAL_CAS [57301:57338]: <ns0:PHOTOMETRIC_MEASUREMENT xmlns:ns0="http:///webanno/custom.ecore" xmlns:ns1="http://www.omg.org/XMI" ns1:id="138023" sofa="1" begin="57301" end="57338" measurement_type="detection" photometric_system="unknown" target="counterpart" certainty="confirmed" magnitude_or_limit="16.2" magnitude_error="0.09" limit_sigma="" unit="mag" photometric_band="" obs_time_raw="85.5" obs_time_type="relative_to_trigger" obs_time_reference="trigger_time_t0" exposure_time_raw="3.0" timezone_raw="" instrument="" comment="missing photometric band; photometric system is unknown" />
  export Priyadarshini [57301:57338]: <ns0:PHOTOMETRIC_MEASUREMENT xmlns:ns0="http:///webanno/custom.ecore" xmlns:ns1="http://www.omg.org/XMI" ns1:id="138023" sofa="1" begin="57301" end="57338" measurement_type="detection" photometric_system="AB" target="counterpart" certainty="confirmed" magnitude_or_limit="16.2" magnitude_error="0.08" limit_sigma="" unit="mag" photometric_band="i'" obs_time_

  export Dahlia [16299:16346]: <ns0:PHOTOMETRIC_MEASUREMENT xmlns:ns0="http:///webanno/custom.ecore" xmlns:ns1="http://www.omg.org/XMI" ns1:id="164011" sofa="1" begin="16299" end="16346" measurement_type="detection" photometric_system="AB" target="counterpart" certainty="confirmed" magnitude_or_limit="17.78" magnitude_error="0.07" limit_sigma="" unit="mag" photometric_band="i" obs_time_raw="2024/10/30 11:32:55" obs_time_type="utc_datetime" obs_time_reference="observation_start" timezone_raw="" instrument="Mephisto" comment="observation time; photometric band; and system were not detcted although its clear in the text. I added the time, band, system, instrument." />


  interim INITIAL_CAS [16299:16346] (1 row(s)):
      document_name layer_source  xmi_id  begin   end                                    covered_text measurement_type photometric_system      target certainty magnitude_or_limit magnitude_error limit_sigma unit photometric_band obs_time_raw obs_time_type obs_time_reference exposure_time_raw timezone_raw instrument                                                                           comment
event_GRB241030.xmi  INITIAL_CAS  164011  16299 16346 2024/10/30 11:32:55  i    79     17.78 +/- 0.07        detection            unknown counterpart confirmed              17.78            0.07              mag                                     unclear            unknown                 i                         missing observation time; missing photometric band; photometric system is unknown
  interim Dahlia [16299:16346] (1 row(s)):
      document_name layer_source  xmi_id  begin   end                                    covered_text measureme

  export Sarah [52089:52096]: <ns0:ASTRO_EVIDENCE xmlns:ns0="http:///webanno/custom.ecore" xmlns:ns1="http://www.omg.org/XMI" ns1:id="122146" sofa="1" begin="52089" end="52096" comment="gcn to be excluded car grandma" />
  interim INITIAL_CAS [52089:52096] (0 row(s)):
    (no row)
  interim Sarah [52089:52096] (1 row(s)):
    document_name layer_source  xmi_id  begin   end covered_text label target certainty value unit                        comment
event_2026owq.xmi        Sarah  122146  52089 52096      GRANDMA  None   None      None  None None gcn to be excluded car grandma
  corpus INITIAL_CAS [52089:52096] (0 row(s)):
    (no row)
  corpus Sarah [52089:52096] (1 row(s)):
    document_name layer_source  xmi_id  begin   end covered_text label target certainty value unit                        comment  span_index match_status changed_fields  is_overlapping  has_category  is_annotator_note comment_status  extractor_vocabulary_gap
event_2026owq.xmi        Sarah  122146  52089 52096    

In [5]:
import json

project = json.loads((EXPORT_ROOT / "exportedproject.json").read_text(encoding="utf-8"))
EXCLUDED_USERS = {"admin", "Patrick", "Thomas"}
export_pairs = sorted({(e["name"], e["user"]) for e in project["annotation_documents"]
                       if e["state"] == "FINISHED" and e["user"] not in EXCLUDED_USERS})
export_documents = sorted({e["name"] for e in project["source_documents"]})


def count_layer(document, stem, layer_local):
    root = read_zip_xmi(document, stem)
    return len(root.findall(qtag(CUSTOM_NS, layer_local)))


evidence_annotator_export = sum(count_layer(d, a, "ASTRO_EVIDENCE") for d, a in export_pairs)
photometry_annotator_export = sum(count_layer(d, a, "PHOTOMETRIC_MEASUREMENT") for d, a in export_pairs)
evidence_baseline_export = sum(count_layer(d, "INITIAL_CAS", "ASTRO_EVIDENCE") for d in export_documents)
photometry_baseline_export = sum(count_layer(d, "INITIAL_CAS", "PHOTOMETRIC_MEASUREMENT") for d in export_documents)

ev_annotator_real = corpus_ev[(corpus_ev["layer_source"] != "INITIAL_CAS") & (corpus_ev["match_status"] != "deleted")]
ev_deleted = corpus_ev[corpus_ev["match_status"] == "deleted"]
ph_annotator_real = corpus_ph[corpus_ph["layer_source"] != "INITIAL_CAS"]
ev_baseline_corpus = corpus_ev[corpus_ev["layer_source"] == "INITIAL_CAS"]
ph_baseline_corpus = corpus_ph[corpus_ph["layer_source"] == "INITIAL_CAS"]
corpus_annotators = pd.read_parquet(CORPUS_DIR / "annotators.parquet")

export_truth = pd.DataFrame([
    {"quantity": "(document, annotator) pairs, state FINISHED", "export": len(export_pairs),
     "corpus": len(corpus_annotators), "expected_difference": "none",
     "explanation": "annotators.parquet is a 1:1 pass-through of the derived perimeter"},
    {"quantity": "ASTRO_EVIDENCE annotations, annotator layers (physical rows only)",
     "export": evidence_annotator_export, "corpus": len(ev_annotator_real), "expected_difference": "none",
     "explanation": "no decision drops or duplicates a real annotation"},
    {"quantity": "ASTRO_EVIDENCE annotator rows including synthetic deleted rows",
     "export": evidence_annotator_export, "corpus": len(ev_annotator_real) + len(ev_deleted),
     "expected_difference": str(len(ev_deleted)),
     "explanation": "decision 3 (adds one synthetic 'deleted' row per baseline span an annotator "
     "did not carry forward)"},
    {"quantity": "PHOTOMETRIC_MEASUREMENT annotations, annotator layers",
     "export": photometry_annotator_export, "corpus": len(ph_annotator_real), "expected_difference": "none",
     "explanation": "decision 3 added 0 synthetic rows in photometry_spans"},
    {"quantity": "ASTRO_EVIDENCE annotations, INITIAL_CAS", "export": evidence_baseline_export,
     "corpus": len(ev_baseline_corpus), "expected_difference": "none",
     "explanation": "decision 13 (INITIAL_CAS retained unchanged)"},
    {"quantity": "PHOTOMETRIC_MEASUREMENT annotations, INITIAL_CAS", "export": photometry_baseline_export,
     "corpus": len(ph_baseline_corpus), "expected_difference": "none",
     "explanation": "decision 13 (INITIAL_CAS retained unchanged)"},
])
print(export_truth.to_string(index=False))

                                                         quantity  export  corpus expected_difference                                                                                        explanation
                      (document, annotator) pairs, state FINISHED      28      28                none                                  annotators.parquet is a 1:1 pass-through of the derived perimeter
ASTRO_EVIDENCE annotations, annotator layers (physical rows only)    4927    4927                none                                                  no decision drops or duplicates a real annotation
   ASTRO_EVIDENCE annotator rows including synthetic deleted rows    4927    4932                   5 decision 3 (adds one synthetic 'deleted' row per baseline span an annotator did not carry forward)
            PHOTOMETRIC_MEASUREMENT annotations, annotator layers    2071    2071                none                                              decision 3 added 0 synthetic rows in photometry_s

In [6]:
def source_sofa(document):
    root = ET.fromstring((EXPORT_ROOT / "source" / document).read_bytes())
    return root.find(qtag(CAS_NS, "Sofa")).get("sofaString")


sofa_by_doc = {d: source_sofa(d) for d in export_documents}

sofa_mismatches = []
for document in export_documents:
    stems = ["INITIAL_CAS"] + [a for d, a in export_pairs if d == document]
    hashes = {hashlib.sha256(sofa_by_doc[document].encode("utf-8")).hexdigest()}
    for stem in stems:
        sofa = read_zip_xmi(document, stem).find(qtag(CAS_NS, "Sofa")).get("sofaString")
        hashes.add(hashlib.sha256(sofa.encode("utf-8")).hexdigest())
    if len(hashes) != 1:
        sofa_mismatches.append(document)

length_by_doc = {d: len(sofa_by_doc[d]) for d in export_documents}
offset_violations = 0
covered_text_mismatches = 0
for df in (corpus_ev, corpus_ph):
    for row in df.itertuples():
        length = length_by_doc[row.document_name]
        if not (0 <= row.begin <= row.end <= length):
            offset_violations += 1
            continue
        if row.covered_text != sofa_by_doc[row.document_name][row.begin:row.end]:
            covered_text_mismatches += 1

total_spans_checked = len(corpus_ev) + len(corpus_ph)
integrity_checks = pd.DataFrame([
    {"check": "sofa byte-identical across every layer of each document",
     "count": f"{len(export_documents)} documents checked",
     "status": "PASS" if not sofa_mismatches else f"FAIL {sofa_mismatches}"},
    {"check": "every span's offsets fall within its document's text length",
     "count": f"{total_spans_checked} spans checked",
     "status": "PASS" if offset_violations == 0 else f"FAIL ({offset_violations})"},
    {"check": "every covered_text equals the export sofa substring at its offsets",
     "count": f"{total_spans_checked} spans checked",
     "status": "PASS" if covered_text_mismatches == 0 else f"FAIL ({covered_text_mismatches})"},
])
print(integrity_checks.to_string(index=False))

                                                             check                count status
           sofa byte-identical across every layer of each document 10 documents checked   PASS
       every span's offsets fall within its document's text length   9356 spans checked   PASS
every covered_text equals the export sofa substring at its offsets   9356 spans checked   PASS


In [7]:
mutation_rows = []
for stage, directory in [("interim", INTERIM_DIR), ("corpus", CORPUS_DIR)]:
    for name in TABLE_NAMES:
        path = directory / f"{name}.parquet"
        current_hash = sha256_of(path)
        original_hash = baseline_fingerprint.loc[
            baseline_fingerprint["file"] == f"{stage}/{name}.parquet", "sha256"].iloc[0]
        mutation_rows.append({"file": f"{stage}/{name}.parquet", "unchanged": current_hash == original_hash})

mutation_check = pd.DataFrame(mutation_rows)
print(mutation_check.to_string(index=False))
assert mutation_check["unchanged"].all(), "a repository file changed during this notebook's run"

shutil.rmtree(TMP_ROOT)
print(f"\nTemporary directory deleted: {not TMP_ROOT.exists()}")

                            file  unchanged
       interim/documents.parquet       True
      interim/annotators.parquet       True
  interim/evidence_spans.parquet       True
interim/photometry_spans.parquet       True
 interim/event_summaries.parquet       True
        corpus/documents.parquet       True
       corpus/annotators.parquet       True
   corpus/evidence_spans.parquet       True
 corpus/photometry_spans.parquet       True
  corpus/event_summaries.parquet       True

Temporary directory deleted: True


## What this test establishes

The corpus is a deterministic function of the INCEpTION export. Running
the two scripts in a clean directory reproduces all ten files byte for
byte, at both stages. Regenerating it requires nothing that is not in
this repository and in the export.

Five annotations were followed from their XML element in the export to
their final row, each crossing different decisions: an identifier that
changes without the content changing, a pair of measurements recorded
over the same text, a correction spanning eight features, a span one
annotator did not carry forward, and a note addressed to the pipeline.
Counts taken directly from the export agree with the corpus, the only
difference being the five synthetic rows decision 3 introduces.

Text integrity was verified across all 9,356 spans: the text each row
records is exactly what its offsets select from the original document,
and the layers of a document share a byte-identical text.